In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/charm-fineval-val/Chinese_Reading_Comprehension.json
/kaggle/input/charm-fineval-val/Chinese_Movie_and_Music_Recommendation.json
/kaggle/input/charm-fineval-val/val_data.json
/kaggle/input/charm-fineval-val/Chinese_Sequence_Understanding.json
/kaggle/input/charm-fineval-val/Chinese_Natural_Language_Inference.json
/kaggle/input/charm-fineval-val/Chinese_Anachronisms_Judgment.json
/kaggle/input/charm-fineval-val/Chinese_Time_Understanding.json
/kaggle/input/charm-fineval-val/Chinese_Sport_Understanding.json
/kaggle/input/fineval-sufe-ant/data/Ant/金融知识/bank_professional.csv
/kaggle/input/fineval-sufe-ant/data/Ant/金融知识/accounting_professional.csv
/kaggle/input/fineval-sufe-ant/data/Ant/金融知识/fund_professional.csv
/kaggle/input/fineval-sufe-ant/data/Ant/金融知识/auditing.csv
/kaggle/input/fineval-sufe-ant/data/Ant/金融知识/security_professional.csv
/kaggle/input/fineval-sufe-ant/data/Ant/金融知识/future_professional.csv
/kaggle/input/fineval-sufe-ant/data/Ant/金融知识/insurance_professional.cs

In [2]:
!pip install deepspeed
!git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
%cd LLaMA-Factory
!pip install -e ".[torch,metrics]"
%cd ..

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 18.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.0/54.0 kB 2.4 MB/s eta 0:00:00
  Created wheel for deepspeed: filename=deepspeed-0.16.5-py3-none-any.whl size=1580581 sha256=044f71c29319c220dce973485a7e1836b6c254c394776fe6a1362c0231622d81
  Stored in directory: /root/.cache/pip/wheels/cb/fa/e7/98efc76db11fac734a4fae8c19dd08cc24257107e132e674f6
Successfully built deepspeed
Cloning into 'LLaMA-Factory'...
remote: Enumerating objects: 353, done.
remote: Counting objects: 100% (353/353), done.
remote: Compressing objects: 100% (281/281), done.
remote: Total 353 (delta 94), reused 183 (delta 57), pack-reused 0 (from 0)
Receiving objects: 100% (353/353), 9.69 MiB | 31.02 MiB/s, done.
Resolving deltas: 100% (94/94), done.
/kaggle/working/LLaMA-Factory
Obtaining file:///kaggle/working/LLaMA-Factory
  Installing build dependencies ... done
  Checking if build backend supports bu

In [3]:
pip install transformers==4.49.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 71.5 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 4.51.1
    Uninstalling transformers-4.51.1:
      Successfully uninstalled transformers-4.51.1
Note: you may need to restart the kernel to use updated packages.


In [4]:
%cd /kaggle/working/LLaMA-Factory
!llamafactory-cli version

/kaggle/working/LLaMA-Factory
2025-04-12 16:24:26.819908: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-04-12 16:24:27.265837: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-04-12 16:24:27.439110: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
[2025-04-12 16:24:45,162] [INFO] [real_accelerator.py:239:get_accelerator] Setting ds_accelerator to cuda (auto detect)
----------------------------------------------------------
| Welcome to LLaMA Factory, version 0.9.3.dev0           |
|                                                        |
| Project page: https://github.com/hiy

In [5]:
import yaml

def update_yaml_config(train_config, parameters_to_update):
    """
    Update parameters in a YAML configuration file.

    Args:
    - train_config (str): Path to the YAML configuration file.
    - parameters_to_update (dict): A dictionary where the keys are the parameter paths 
      (e.g., "model.layers") and the values are the new values to set.

    Returns:
    - None: The function updates the YAML file directly.
    """
    
    def update_nested_dict(data, key_path, value):
        """
        Update a nested dictionary with the specified key path and value.
        key_path: A string representing the path to the key, e.g., "model.layers".
        value: The new value to set.
        """
        keys = key_path.split(".")
        d = data
        for key in keys[:-1]:
            if key not in d:
                d[key] = {}
            d = d[key]
        d[keys[-1]] = value

    try:
        with open(train_config, 'r', encoding='utf-8') as file:
            data = yaml.safe_load(file)

        # Update parameters in the YAML file
        for parameter_name, new_value in parameters_to_update.items():
            if "." in parameter_name:
                update_nested_dict(data, parameter_name, new_value)
            elif parameter_name in data:
                data[parameter_name] = new_value
            else:
                print(f"Warning: The parameter '{parameter_name}' does not exist in the YAML file.")

        # Print the updated content (optional)
        print(yaml.dump(data, default_flow_style=False, allow_unicode=True))

        with open(train_config, 'w', encoding='utf-8') as file:
            yaml.safe_dump(data, file, default_flow_style=False, allow_unicode=True)

    except Exception as e:
        print(f"Error: {e}")


In [6]:
!pip list

Package                            Version              Editable project location
---------------------------------- -------------------- -----------------------------
absl-py                            1.4.0
accelerate                         1.2.1
aiofiles                           22.1.0
aiohappyeyeballs                   2.4.6
aiohttp                            3.11.12
aiosignal                          1.3.2
aiosqlite                          0.21.0
alabaster                          1.0.0
albucore                           0.0.19
albumentations                     1.4.20
alembic                            1.14.1
altair                             5.5.0
annotated-types                    0.7.0
annoy                              1.17.3
ansicolors                         1.1.8
antlr4-python3-runtime             4.9.3
anyio                              4.9.0
argon2-cffi                        23.1.0
argon2-cffi-bindings               21.2.0
args                               0.1.0
ar

In [7]:
!nvidia-smi

Sat Apr 12 16:24:57 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 560.35.03              Driver Version: 560.35.03      CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             10W /   70W |       1MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Data Preparation

In [8]:
import pandas as pd
import numpy as np
import os
filenames = os.listdir("/kaggle/input/fineval-sufe-ant/data/SUFE/val")
subject_list = [val_file.replace("_val.csv","") for val_file in filenames]

dataset = []
def convert_to_dict(row):
    if 'explaination' in row.index:
        exp = row['explaination'] + '正确答案是：'
    else:
        exp = ''
    if str(row['answer']) == 'nan':
        return None
    
    return {
            "subject":str(subject_name),
            "conversation":[
                {"from": "human", "value": '以下是中国金融考试的单项选择题，请选出其中的正确答案。\n'+row['question']+' A:'+str(row['A'])+' B:'+str(row['B'])+' C:'+str(row['C'])+' D:'+str(row['D'])},
                {"from": "gpt", "value": exp+str(row['answer'])}
            ],
            "ground_truth":row['answer'],
        }
print('preparing dataset...')

for index,subject_name in enumerate(subject_list):
    val_file_path=os.path.join('/kaggle/input/fineval-sufe-ant/data/SUFE/val', f'{subject_name}_val.csv')
    val_df=pd.read_csv(val_file_path)
    val_df = val_df.apply(convert_to_dict, axis=1).dropna().tolist()
    dataset.extend(val_df)
    print(f"Processed {index + 1}/{len(subject_list)}: {subject_name} - {len(val_df)} samples added")

filenames = os.listdir("/kaggle/input/fineval-sufe-ant/data/Ant/金融知识")
subject_list = [val_file.replace(".csv","") for val_file in filenames]

print('preparing dataset...')
for index,subject_name in enumerate(subject_list):
    val_file_path=os.path.join('/kaggle/input/fineval-sufe-ant/data/Ant/金融知识', f'{subject_name}.csv')
    val_df=pd.read_csv(val_file_path)
    val_df = val_df.apply(convert_to_dict, axis=1).dropna().tolist()
    dataset.extend(val_df)
    print(f"Processed {index + 1}/{len(subject_list)}: {subject_name} - {len(val_df)} samples added")

preparing dataset...
Processed 1/32: auditing - 32 samples added
Processed 2/32: banking_practitioner_qualification_certificate - 116 samples added
Processed 3/32: advanced_financial_accounting - 21 samples added
Processed 4/32: financial_markets - 39 samples added
Processed 5/32: china_actuary - 37 samples added
Processed 6/32: finance - 25 samples added
Processed 7/32: econometrics - 18 samples added
Processed 8/32: economic_law - 25 samples added
Processed 9/32: commercial_bank_finance - 20 samples added
Processed 10/32: financial_management - 24 samples added
Processed 11/32: management_accounting - 29 samples added
Processed 12/32: corporate_finance - 36 samples added
Processed 13/32: financial_engineering - 26 samples added
Processed 14/32: investments - 38 samples added
Processed 15/32: futures_practitioner_qualification_certificate - 39 samples added
Processed 16/32: central_banking - 28 samples added
Processed 17/32: monetary_finance - 43 samples added
Processed 18/32: interna

In [9]:
# 添加dev中的170条数据,注意dataset_cot已包含无cot的题目
filenames = os.listdir("/kaggle/input/fineval-sufe-ant/data/SUFE/dev")
subject_list = [val_file.replace("_dev.csv","") for val_file in filenames]
dataset_cot = dataset.copy()
print('preparing dataset with CoT...')
for index,subject_name in enumerate(subject_list):
    val_file_path=os.path.join('/kaggle/input/fineval-sufe-ant/data/SUFE/dev', f'{subject_name}_dev.csv')
    val_df=pd.read_csv(val_file_path)
    val_df = val_df.apply(convert_to_dict, axis=1).dropna().tolist()
    dataset_cot.extend(val_df)
    print(f"Processed {index + 1}/{len(subject_list)}: {subject_name} - {len(val_df)} samples added")

preparing dataset with CoT...
Processed 1/34: certified_practising_accountant - 5 samples added
Processed 2/34: financial_management - 5 samples added
Processed 3/34: statistics - 5 samples added
Processed 4/34: certified_management_accountant - 5 samples added
Processed 5/34: financial_markets - 5 samples added
Processed 6/34: public_finance - 5 samples added
Processed 7/34: central_banking - 5 samples added
Processed 8/34: banking_practitioner_qualification_certificate - 5 samples added
Processed 9/34: monetary_finance - 5 samples added
Processed 10/34: macroeconomics - 5 samples added
Processed 11/34: management_accounting - 5 samples added
Processed 12/34: securities_practitioner_qualification_certificate - 5 samples added
Processed 13/34: auditing - 5 samples added
Processed 14/34: political_economy - 5 samples added
Processed 15/34: accounting - 5 samples added
Processed 16/34: economic_law - 5 samples added
Processed 17/34: china_actuary - 5 samples added
Processed 18/34: corpor

In [10]:
filenames = os.listdir("/kaggle/input/charm-fineval-val")
subject_list = [val_file.replace('.json','') for val_file in filenames if val_file.startswith('Chinese_')]

import json

def transform_data(subject):
    with open('/kaggle/input/charm-fineval-val/'+subject+'.json', "r", encoding="utf-8") as f:
        data = json.load(f)

    transformed_data = []
    for example in data['examples']:
        subject_name = subject  
        question = example['input']
        answer = example['target'].replace('(', '').replace(')', '')
        
        new_format = {
            "subject":subject_name,
            "conversation":[
                {"from": "human", "value": '以下是中国常识的单项选择题，请选出其中的正确答案。\n'+question},
                {"from": "gpt", "value": answer}
            ],
            "ground_truth":answer,
        }
        transformed_data.append(new_format)
    
    return transformed_data

print('preparing dataset...')
dataset_CHARM = []
for index,subject in enumerate(subject_list):
    data = transform_data(subject)
    dataset_CHARM.extend(data)
    print(f"Processed {index + 1}/{len(subject_list)}: {subject} - {len(data)} samples added")

with open("CHRAM.json", "w", encoding="utf-8") as f:
    json.dump(dataset_CHARM, f, ensure_ascii=False, indent=4)

preparing dataset...
Processed 1/7: Chinese_Reading_Comprehension - 200 samples added
Processed 2/7: Chinese_Movie_and_Music_Recommendation - 50 samples added
Processed 3/7: Chinese_Sequence_Understanding - 100 samples added
Processed 4/7: Chinese_Natural_Language_Inference - 100 samples added
Processed 5/7: Chinese_Anachronisms_Judgment - 150 samples added
Processed 6/7: Chinese_Time_Understanding - 100 samples added
Processed 7/7: Chinese_Sport_Understanding - 200 samples added


In [11]:
# 整合所有样本数据(Fineval+cot+CHARM)
# 随机分割数据集 4:1
import torch
import json
from torch.utils.data import random_split, Dataset, DataLoader,ConcatDataset

# 设置随机种子
torch.manual_seed(42)
train_size = int(0.8 * len(dataset_cot))
val_size = len(dataset_cot) - train_size
train_dataset_full, val_dataset_full = random_split(dataset_cot, [train_size, val_size]) # torch.dataset format

# 提取数据list格式待后续使用
train_data = [dataset_cot[i] for i in train_dataset_full.indices]

torch.manual_seed(42)
train_size = int(0.8 * len(dataset_CHARM))
val_size = len(dataset_CHARM) - train_size
train_dataset_CHARM, val_dataset_CHARM = random_split(dataset_CHARM, [train_size, val_size]) # torch.dataset format

train_dataset_full = ConcatDataset([train_dataset_full, train_dataset_CHARM])
train_data = train_data+[dataset_CHARM[i] for i in train_dataset_CHARM.indices]

val_data = [dataset_cot[i] for i in val_dataset_full.indices]
val_data_CHARM = [dataset_CHARM[i] for i in val_dataset_CHARM.indices]

print(val_data_CHARM[0]['conversation'])
print(val_data_CHARM[0]['ground_truth'])

[{'from': 'human', 'value': '以下是中国常识的单项选择题，请选出其中的正确答案。\n以下陈述是否包含时代错误，请选择正确选项。一个接受了义务教育、具备基本常识的人会如何选择？唐玄宗在颐和园中赏花。\n选项：\n(A) 是 \n(B) 否'}, {'from': 'gpt', 'value': 'A'}]
A


In [12]:
# 导出为 JSON 文件
import json
with open("dataset.json", "w", encoding="utf-8") as f:
    json.dump(dataset, f, ensure_ascii=False, indent=4)
print("Fin dataset JSON 文件已导出到 dataset.json")

with open("dataset_cot.json", "w", encoding="utf-8") as f:
    json.dump(dataset_cot, f, ensure_ascii=False, indent=4)
print("Fin+CoT dataset JSON 文件已导出到 dataset_cot.json")

with open("dataset_CHARM.json", "w", encoding="utf-8") as f:
    json.dump(dataset_CHARM, f, ensure_ascii=False, indent=4)
print("CHARM dataset JSON 文件已导出到 dataset_CHARM.json")

with open("train_dataset_full.json", "w", encoding="utf-8") as f:
    json.dump(train_data, f, ensure_ascii=False, indent=4)
print("Full train dataset JSON 文件已导出到 train_dataset_full.json")

with open("val_data.json", "w", encoding="utf-8") as f:
    json.dump(val_data, f, ensure_ascii=False, indent=4)
print("Fin validation data JSON 文件已导出到 val_data.json")

with open("val_data_CHARM.json", "w", encoding="utf-8") as f:
    json.dump(val_data_CHARM, f, ensure_ascii=False, indent=4)
print("CHARM validation data JSON 文件已导出到 val_data_CHARM.json")

Fin dataset JSON 文件已导出到 dataset.json
Fin+CoT dataset JSON 文件已导出到 dataset_cot.json
CHARM dataset JSON 文件已导出到 dataset_CHARM.json
Full train dataset JSON 文件已导出到 train_dataset_full.json
Fin validation data JSON 文件已导出到 val_data.json
CHARM validation data JSON 文件已导出到 val_data_CHARM.json


In [13]:
# 保存并登记 training dataset到 LLama-Factory中
import json
%cd /kaggle/working/LLaMA-Factory
# 数据保存
with open("data/train_dataset_full.json", "w", encoding="utf-8") as f:
  json.dump(train_data, f, indent=2, ensure_ascii=False)
with open("data/val_data.json", "w", encoding="utf-8") as f:
  json.dump(val_data, f, indent=2, ensure_ascii=False)
with open("data/val_data_CHARM.json", "w", encoding="utf-8") as f:
  json.dump(val_data_CHARM, f, indent=2, ensure_ascii=False)

# 数据登记
NAME = "FinGPT"
AUTHOR = "ZHOUYixin"

# 模型登记
with open("data/identity.json", "r", encoding="utf-8") as f:
  dataset = json.load(f)

for sample in dataset:
  sample["output"] = sample["output"].replace("{{"+ "name" + "}}", NAME).replace("{{"+ "author" + "}}", AUTHOR)

with open("data/identity.json", "w", encoding="utf-8") as f:
  json.dump(dataset, f, indent=2, ensure_ascii=False)

# 数据集登记
file_path = "./data/dataset_info.json"
with open(file_path, "r", encoding="utf-8") as f:
    data_info = json.load(f)

new_data = {
    "train_dataset_full": {
        "file_name": "train_dataset_full.json",
        "formatting":"sharegpt",
        "columns":{
            "message":"conversation"
        }
    },
    "val_data":{
        "file_name": "val_data.json",
        "formatting":"sharegpt",
        "columns":{
            "message":"conversation"
        }
    },
    "val_data_CHARM":{
        "file_name": "val_data_CHARM.json",
        "formatting":"sharegpt",
        "columns":{
            "message":"conversation"
        }
    },
}
data_info.update(new_data)

with open(file_path, "w", encoding="utf-8") as f:
    json.dump(data_info, f, indent=2, ensure_ascii=False)

/kaggle/working/LLaMA-Factory


## SFT

In [14]:
# set train config
train_config = "examples/train_qlora/llama3_lora_sft_otfq.yaml"
parameters_to_update = {
### model
"model_name_or_path": "Qwen/Qwen2-7B-Instruct",
# "model_name_or_path": model0,

### method
"stage": "sft",
"do_train": True,
"finetuning_type": "lora",
"lora_target": 'all',
"lora_dropout": 0,
"lora_rank": 8,
"deepspeed": "examples/deepspeed/ds_z0_config.json",

### dataset
"dataset": 'train_dataset_full',
#"template": "Alpaca",
"cutoff_len": 2048,
"max_samples": 400000,
"overwrite_cache": True,
"preprocessing_num_workers": 1,

### output
"output_dir": "saves/qlora",
"logging_steps": 1,
"save_steps": 0,
"plot_loss": True,
"overwrite_output_dir": True,

### train
"per_device_train_batch_size": 2,
"gradient_accumulation_steps": 2,
"learning_rate": 2.0e-7,
"num_train_epochs": 1.0,
"lr_scheduler_type": "cosine",
"warmup_ratio": 0.03,
"bf16": True,
"ddp_timeout": 180000000,

### eval
"val_size": 0.01, # 暂不用自带eval功能
"per_device_eval_batch_size": 2,
"eval_strategy": "steps",
"eval_steps": 200
}
update_yaml_config(train_config, parameters_to_update)

!wandb disabled
!FORCE_TORCHRUN=1 llamafactory-cli train examples/train_qlora/llama3_lora_sft_otfq.yaml

bf16: true
cutoff_len: 2048
dataloader_num_workers: 4
dataset: train_dataset_full
ddp_timeout: 180000000
do_train: true
finetuning_type: lora
gradient_accumulation_steps: 2
learning_rate: 2.0e-07
logging_steps: 1
lora_rank: 8
lora_target: all
lr_scheduler_type: cosine
max_samples: 400000
model_name_or_path: Qwen/Qwen2-7B-Instruct
num_train_epochs: 1.0
output_dir: saves/qlora
overwrite_cache: true
overwrite_output_dir: true
per_device_train_batch_size: 2
plot_loss: true
preprocessing_num_workers: 1
quantization_bit: 4
quantization_method: bnb
report_to: none
save_only_model: false
save_steps: 0
stage: sft
template: llama3
trust_remote_code: true
warmup_ratio: 0.03

W&B disabled.
2025-04-12 16:25:07.351142: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-04-12 16:25:07.372473: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to r

## Merge Sft

In [15]:
# # set train config
# train_config = "examples/merge_lora/gemma2_lora_sft_export.yaml"
# parameters_to_update = {
# ### model
# "model_name_or_path": "/kaggle/input/gemma2-2b-it-pretrain-medical/transformers/default/1/gemma2_lora_pretrain_medical",
# "adapter_name_or_path": "saves/gemma2_2b_it/lora/sft_afterpretrain_medicalq&a",
# "template": "gemma",
# "finetuning_type": "lora",

# ### export
# "export_dir": "models/gemma2_lora_pretrain_sft_medical_phase1",
# "export_size": 2,
# "export_device": "cpu",
# "export_legacy_format": False
# }
# update_yaml_config(train_config, parameters_to_update)

# !wandb disabled
# !FORCE_TORCHRUN=1 llamafactory-cli export examples/merge_lora/gemma2_lora_sft_export.yaml